# MODUL PRAKTIKUM BIG DATA
## Pertemuan 11 — Machine Learning dengan Spark MLlib: Feature Engineering & Regresi

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 11 |
| **Topik** | Pengenalan Spark MLlib, Feature Engineering, Regresi Linear Skala Besar |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-10 selesai (Spark/PySpark 3.5.9 terpasang, memahami DataFrame, join, ETL) |

---

### Cara Menggunakan Modul Ini

**Tidak ada instalasi baru** pada pertemuan ini — **Spark MLlib** (modul `pyspark.ml`) adalah bagian bawaan dari paket Spark 3.5.9 yang sudah anda pasang sejak Pertemuan 4, persis seperti Structured Streaming pada Pertemuan 10. Seluruh modul dikerjakan langsung di Jupyter Notebook.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11** — tidak ada perubahan versi apa pun. Modul ini memperkenalkan API baru (`pyspark.ml`) dari package yang sama.


---
## Recap Pertemuan Sebelumnya

- [ ] Memahami `SparkSession`, DataFrame, `groupBy`/`agg`, join dari Pertemuan 4-7
- [ ] Memahami konsep batch vs streaming dari Pertemuan 9-10 (tidak dipakai langsung hari ini)

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 11, mahasiswa mampu:
1. Menjelaskan mengapa pemrosesan Machine Learning skala besar membutuhkan pendekatan berbeda dari scikit-learn biasa.
2. Menyiapkan data (feature engineering) menggunakan `StringIndexer` dan `VectorAssembler`.
3. Melatih model regresi linear menggunakan Spark MLlib.
4. Mengevaluasi performa model regresi menggunakan RMSE dan R².

---

## 11.1 Mengapa Machine Learning Butuh "Versi Big Data"?

Banyak dari anda mungkin sudah pernah menggunakan **scikit-learn** pada mata kuliah lain untuk melatih model Machine Learning. Scikit-learn sangat baik untuk dataset yang **muat dalam memori (RAM) satu komputer** — persis seperti keterbatasan pandas yang kita bahas di Pertemuan 3. Ketika data pelatihan berjumlah puluhan juta baris hingga miliaran (tersebar di banyak node HDFS, misalnya), scikit-learn tidak lagi memadai.

**Spark MLlib** (modul `pyspark.ml`) menyediakan algoritma Machine Learning yang dirancang untuk **berjalan terdistribusi** — data pelatihan dapat tersebar di banyak executor sekaligus, dan proses pelatihan model memanfaatkan seluruh cluster, bukan hanya satu mesin. API-nya sengaja dirancang **mirip scikit-learn** (konsep `fit()` dan `transform()`/`predict()`) agar mudah dipelajari oleh siapa pun yang sudah terbiasa dengan scikit-learn.

| Aspek | scikit-learn | Spark MLlib |
|---|---|---|
| **Skala data** | Harus muat di RAM satu komputer | Terdistribusi di banyak node |
| **Struktur data input** | NumPy array / pandas DataFrame | Spark DataFrame (kolom `features` berformat Vector) |
| **Paralelisasi** | Umumnya satu mesin (beberapa core) | Banyak mesin sekaligus (cluster) |
| **Filosofi API** | `fit()`, `predict()` | `fit()`, `transform()` — sangat mirip! |

> Pada praktikum ini kita tetap menjalankan Spark dalam mode `local[*]` (satu VM), namun **kode yang sama persis** dapat dijalankan di atas cluster Hadoop/YARN real (seperti yang telah anda pasang sejak Pertemuan 3) tanpa perlu mengubah satu baris kode pun — cukup mengubah konfigurasi cluster manager saat menjalankan.


---
## 11.2 Persiapan: SparkSession & Dataset

Kita akan melanjutkan skenario e-commerce yang sudah sering kita pakai — kali ini membangun model untuk **memprediksi total pendapatan sebuah transaksi** berdasarkan atribut-atributnya (jumlah unit, harga satuan, kategori, kota, dan rating).

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
# Total pendapatan dipengaruhi unit & harga (dengan sedikit variasi acak, seperti data real world)
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
df.show(5)
print("Total baris:", df.count())

+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows

Total baris: 2000


---
## 11.3 Feature Engineering: Menyiapkan Data untuk MLlib

Berbeda dari scikit-learn yang menerima banyak kolom terpisah, **hampir seluruh algoritma Spark MLlib mewajibkan seluruh fitur digabung menjadi SATU kolom bertipe `Vector`**, biasanya dinamai `features`. Selain itu, kolom kategorikal (teks) harus diubah menjadi angka terlebih dahulu — model Machine Learning tidak dapat memproses teks secara langsung.

### 11.3.1 `StringIndexer` — Mengubah Kategori Menjadi Angka

In [6]:
from pyspark.ml.feature import StringIndexer

# StringIndexer mengubah nilai kategori (teks) menjadi indeks numerik
# berdasarkan frekuensi kemunculan (nilai paling sering muncul -> indeks 0, dst.)
indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

# Perhatikan pola dua tahap: fit() dulu (mempelajari daftar kategori), baru transform() (menerapkannya)
df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

df_indexed.select("kategori", "kategori_idx", "kota", "kota_idx").show(8)

+------------+------------+----------+--------+
|    kategori|kategori_idx|      kota|kota_idx|
+------------+------------+----------+--------+
|     Fashion|         4.0|  Magelang|     1.0|
|Rumah Tangga|         1.0|  Semarang|     2.0|
|  Elektronik|         3.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|  Magelang|     1.0|
|     Makanan|         0.0|  Semarang|     2.0|
|   Kesehatan|         2.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|      Solo|     3.0|
|   Kesehatan|         2.0|  Semarang|     2.0|
+------------+------------+----------+--------+
only showing top 8 rows



> **Pola `fit()` lalu `transform()`** akan sangat sering muncul di Spark MLlib — `fit()` "mempelajari" sesuatu dari data (di sini: daftar kategori unik beserta urutan frekuensinya), menghasilkan objek **Model** yang kemudian dipakai untuk `transform()` (menerapkan hasil pembelajaran tadi ke data). Pola yang identik akan kita pakai lagi saat melatih model regresi itu sendiri di Sub-bab 11.4.

### 11.3.2 `VectorAssembler` — Menggabungkan Kolom Menjadi Satu Fitur

In [7]:
from pyspark.ml.feature import VectorAssembler

# Menggabungkan seluruh kolom numerik (termasuk hasil StringIndexer) menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "pendapatan").show(5, truncate=False)

+---------------------------+------------------+
|features                   |pendapatan        |
+---------------------------+------------------+
|[7.0,150000.0,4.0,1.0,4.0] |997913.0612957058 |
|[4.0,100000.0,1.0,2.0,4.0] |389490.8317158621 |
|[13.0,150000.0,3.0,0.0,4.0]|1936047.6709878189|
|[11.0,100000.0,1.0,1.0,5.0]|1080918.9109179082|
|[8.0,150000.0,0.0,2.0,2.0] |1191696.4943767258|
+---------------------------+------------------+
only showing top 5 rows



Perhatikan kolom `features` sekarang berisi satu **vector** yang merangkum kelima kolom input kita — inilah format yang dibutuhkan hampir seluruh algoritma Spark MLlib.


---
## 11.4 Melatih Model Regresi Linear

### 11.4.1 Membagi Data: Training vs Testing

Sama seperti prinsip Machine Learning pada umumnya (termasuk di scikit-learn): kita **tidak boleh** testing model dengan data yang sama persis dipakai untuk training — ini akan memberi gambaran performa yang terlalu optimis dan menyesatkan.

In [8]:
# randomSplit membagi data secara acak — 80% untuk latihan (training), 20% untuk pengujian (testing)
train, test = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

Jumlah data training : 1608
Jumlah data testing   : 392


> **Parameter `seed=42`** memastikan pembagian data selalu identik setiap kali kode dijalankan ulang — penting agar hasil eksperimen dapat direproduksi dan dibandingkan secara adil (persis seperti `np.random.seed()` yang sudah sering kita pakai sejak Pertemuan 2).

### 11.4.2 Melatih Model

In [9]:
from pyspark.ml.regression import LinearRegression

# Mendefinisikan model: featuresCol menunjuk ke kolom fitur, labelCol menunjuk ke kolom target yang diprediksi
lr = LinearRegression(featuresCol="features", labelCol="pendapatan")

# fit() melatih model menggunakan data training -- inilah tahap "pembelajaran" sesungguhnya
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Model berhasil dilatih!
Koefisien tiap fitur: [82485.42527432049,7.628834502549623,1917.9750235245292,3459.554426735934,-3020.0236907697836]
Intercept: -623170.8077932554


**Koefisien** menunjukkan seberapa besar pengaruh tiap fitur terhadap prediksi — urutannya mengikuti urutan `inputCols` pada `VectorAssembler` (`unit_terjual`, `harga_satuan`, `kategori_idx`, `kota_idx`, `rating`). Semakin besar nilai absolut koefisien suatu fitur, semakin besar pengaruhnya terhadap prediksi pendapatan (dengan catatan skala satuan tiap fitur perlu diperhatikan saat membandingkan).

### 11.4.3 Membuat Prediksi

In [10]:
predictions = model.transform(test)
predictions.select("unit_terjual", "harga_satuan", "kategori", "kota", "pendapatan", "prediction").show(10)

+------------+------------+------------+----------+------------------+-------------------+
|unit_terjual|harga_satuan|    kategori|      kota|        pendapatan|         prediction|
+------------+------------+------------+----------+------------------+-------------------+
|           1|       25000|Rumah Tangga|  Magelang|64514.717361452844|-350627.03788647347|
|           1|       50000|     Fashion|      Solo|38256.916819375925|-156293.21247099683|
|           1|       50000|   Kesehatan|Yogyakarta| 73585.18669220252|-161447.75472594437|
|           1|       75000|     Fashion|  Semarang| 70325.89976583082|  43048.19042908691|
|           1|      100000|Rumah Tangga|  Magelang|104987.40565455305| 221535.54980474827|
|           1|      150000|   Kesehatan|  Magelang| 190361.5476543564|  598855.2025742142|
|           2|       25000|     Makanan|Yogyakarta| 76667.74600440849| -273519.1420624135|
|           2|       75000|     Makanan|      Solo|  176018.830260634| 115281.22265450563|

Kolom `prediction` adalah hasil tebakan model, dibandingkan dengan `pendapatan` (nilai sesungguhnya). Mari ukur seberapa akurat tebakan ini secara keseluruhan.

---

## 11.5 Evaluasi Model Regresi

In [11]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE (Root Mean Squared Error) -- rata-rata besar kesalahan prediksi, dalam satuan yang sama dengan target (Rupiah)
evaluator_rmse = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# R-squared -- seberapa besar proporsi variasi data yang berhasil "dijelaskan" oleh model (0 = buruk, 1 = sempurna)
evaluator_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 184,363
R²   : 0.8593  (85.9% variasi data berhasil dijelaskan model)


| Metrik | Interpretasi |
|---|---|
| **RMSE** | Rata-rata kesalahan prediksi dalam satuan asli (Rupiah). Semakin **kecil**, semakin baik. |
| **R² (R-squared)** | Proporsi variasi data yang dijelaskan model, skala 0-1. Semakin **mendekati 1**, semakin baik. |

R² di atas 0.8 pada data simulasi kita menunjukkan model cukup baik menangkap pola hubungan antara fitur-fitur dan pendapatan — masuk akal, karena kita memang merancang `pendapatan` sebagian besar berdasarkan `unit_terjual x harga_satuan` (ditambah sedikit variasi acak) saat membuat dataset simulasi tadi.

---

## Menutup SparkSession

In [12]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan dataset (Sub-bab 11.2) sebelum mengerjakan latihan berikut.

In [14]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
print("Siap untuk latihan. Total baris:", df.count())

Siap untuk latihan. Total baris: 2000


**Soal 1.** Ulangi proses `StringIndexer` untuk kolom `kategori` dan `kota`, lalu buat `VectorAssembler` **baru** yang HANYA menggunakan tiga fitur: `unit_terjual`, `harga_satuan`, dan `rating` (tanpa `kategori_idx`/`kota_idx`). Beri nama kolom hasil `features_sederhana`.

In [18]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

indexer_kategori = indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "rating"],
    outputCol="features")

df_final = assembler.transform(df_indexed)
df_final.select("unit_terjual", "harga_satuan", "rating", "features").show(5, truncate=False)

+------------+------------+------+-------------------+
|unit_terjual|harga_satuan|rating|features           |
+------------+------------+------+-------------------+
|7           |150000      |4     |[7.0,150000.0,4.0] |
|4           |100000      |4     |[4.0,100000.0,4.0] |
|13          |150000      |4     |[13.0,150000.0,4.0]|
|11          |100000      |5     |[11.0,100000.0,5.0]|
|8           |150000      |2     |[8.0,150000.0,2.0] |
+------------+------------+------+-------------------+
only showing top 5 rows



**Soal 2.** Latih model `LinearRegression` baru menggunakan `features_sederhana` dari Soal 1 (gunakan train/test split yang sama, `seed=42`). Bandingkan nilai **R²** model ini dengan model utama pada Sub-bab 11.4 yang menggunakan 5 fitur — apakah performanya lebih baik, sama, atau lebih buruk?

In [24]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Membagi data training dan testing
train, test = df_final.randomSplit(
    [0.8, 0.2], seed=42)

# Membuat model regresi linear
lr_sederhana = LinearRegression(
    featuresCol="features",
    labelCol="pendapatan")

model = lr.fit(train)

# Melakukan prediksi
prediksi_sederhana = model.transform(test)

# Menghitung R²
evaluator = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2_sederhana = evaluator.evaluate(prediksi_sederhana)

# Membandingkan dengan model utama
prediksi_utama = model.transform(test)
r2_utama = evaluator.evaluate(prediksi_utama)

print("R² Model Utama (5 fitur):", r2_utama)
print("R² Model Sederhana (3 fitur):", r2_sederhana)

if r2_sederhana > r2_utama:
    print("Model sederhana memiliki performa lebih baik.")
elif r2_sederhana < r2_utama:
    print("Model sederhana memiliki performa lebih buruk.")
else:
    print("Kedua model memiliki performa yang sama.")


R² Model Utama (5 fitur): 0.8592283136869068
R² Model Sederhana (3 fitur): 0.8592283136869068
Kedua model memiliki performa yang sama.


**Soal 3.** Menggunakan model utama (5 fitur) dari Sub-bab 11.4, tampilkan **5 baris prediksi dengan selisih (`pendapatan - prediction`) terbesar** — yaitu prediksi yang paling meleset. (Petunjuk: tambahkan kolom baru berisi nilai absolut selisihnya, lalu urutkan.)

In [26]:
from pyspark.sql.functions import abs, desc

# Melakukan prediksi menggunakan model utama
prediksi_utama = model.transform(test)

# Menghitung selisih absolut
prediksi_error = prediksi_utama.withColumn("selisih_absolut", abs(col("pendapatan") - col("prediction")))

# Menampilkan 5 prediksi dengan kesalahan terbesar
prediksi_error.orderBy(desc("selisih_absolut")).select("unit_terjual", "harga_satuan", "kategori", "kota", "pendapatan", "prediction","selisih_absolut").show(5, truncate=False)


+------------+------------+------------+----------+------------------+------------------+------------------+
|unit_terjual|harga_satuan|kategori    |kota      |pendapatan        |prediction        |selisih_absolut   |
+------------+------------+------------+----------+------------------+------------------+------------------+
|1           |150000      |Makanan     |Magelang  |110414.0339392634 |600304.1773523054 |489890.14341304195|
|1           |150000      |Makanan     |Yogyakarta|120877.75807042063|600304.1773523054 |479426.41928188474|
|14          |150000      |Kesehatan   |Magelang  |2128127.40810033  |1675598.0416954895|452529.36640484026|
|14          |150000      |Rumah Tangga|Yogyakarta|2128292.853349907 |1678580.3726891214|449712.48066078546|
|1           |150000      |Elektronik  |Solo      |171411.58935156555|609251.1703332005 |437839.5809816349 |
+------------+------------+------------+----------+------------------+------------------+------------------+
only showing top 5 

**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa Spark MLlib mewajibkan kita menggabungkan seluruh fitur menjadi satu kolom `Vector` (melalui `VectorAssembler`), bukan menerima banyak kolom terpisah seperti kebiasaan di pandas/scikit-learn? Tulis jawaban pada markdown cell di bawah ini.

Spark MLlib menggunakna satu kolom Vektor agar seluruh fitur memiliki format yang seragam sehingga dapat diproses oleh algoritma Machien Learning dengan lebih mudah. VectorAssembler digunakan untuk menggabungkan beberapa kolom Fitur menjadi satu kompulan data numerik yang dibutuhkan dalam proses pelatihan dan prediksi model.


In [27]:
spark.stop
print("SparkSEssion ditutup.")

SparkSEssion ditutup.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas11_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Feature Engineering | `StringIndexer` & `VectorAssembler` diterapkan dengan benar | 20% |
| B. Split & Pelatihan | Split proporsional benar, model berhasil dilatih | 20% |
| C. Evaluasi | RMSE & R² dihitung benar, interpretasi masuk akal | 20% |
| D. Eksperimen Perbandingan | Model kedua benar, perbandingan & kesimpulan logis | 25% |
| E. Prediksi Skenario Baru | Ketiga skenario diproses & diprediksi dengan benar | 15% |
